# NYC Citi Bike — Linear Regression

## Objective
Predict `trip_duration_min` using Linear Regression.

## Data
Processed dataset:
`../data/processed/citibike_final_cleaned.csv`

## ML Workflow
1. Load cleaned data
2. Select features and target
3. Check missing values
4. Sort by time
5. Time-based train/test split
6. Mean baseline
7. Linear Regression
8. Model evaluation
9. Coefficient analysis
10. Actual vs predicted
11. Residual analysis


In [19]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [20]:
# Load cleaned dataset

df = pd.read_csv("../data/processed/citibike_final_cleaned.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


C:\Users\Ritesh\AppData\Local\Temp\ipykernel_22044\2529480524.py:3: DtypeWarning: Columns (0: start_station_id, 1: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/citibike_final_cleaned.csv")


Shape: (5244973, 24)

Columns:
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'trip_duration_min', 'trip_distance_km', 'member_casual', 'start_station_id', 'start_station_name', 'start_lat', 'start_lng', 'start_capacity', 'end_station_id', 'end_station_name', 'end_lat', 'end_lng', 'end_capacity', 'hour', 'day_of_week', 'is_weekend', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']


In [21]:
# Basic inspection

print(df.info())

print("\nMissing values:")
print(df.isnull().sum())


<class 'pandas.DataFrame'>
RangeIndex: 5244973 entries, 0 to 5244972
Data columns (total 24 columns):
 #   Column                Dtype  
---  ------                -----  
 0   ride_id               str    
 1   rideable_type         str    
 2   started_at            str    
 3   ended_at              str    
 4   trip_duration_min     float64
 5   trip_distance_km      float64
 6   member_casual         str    
 7   start_station_id      object 
 8   start_station_name    str    
 9   start_lat             float64
 10  start_lng             float64
 11  start_capacity        float64
 12  end_station_id        object 
 13  end_station_name      str    
 14  end_lat               float64
 15  end_lng               float64
 16  end_capacity          float64
 17  hour                  int64  
 18  day_of_week           str    
 19  is_weekend            bool   
 20  temperature_2m        float64
 21  relative_humidity_2m  float64
 22  precipitation         float64
 23  wind_speed_10m    

In [22]:
# Convert timestamp

df['started_at'] = pd.to_datetime(df['started_at'])

# Sort chronologically
df = df.sort_values('started_at').reset_index(drop=True)

print("Data period:")
print(df['started_at'].min(), "to", df['started_at'].max())


Data period:
2026-07-31 12:07:25.382000 to 2026-08-31 23:58:01.765000


## 1. Features and Target

Target:
`trip_duration_min`

Initial features:
- `hour`
- `is_weekend`
- `trip_distance_km`
- `temperature_2m`
- `relative_humidity_2m`
- `precipitation`
- `wind_speed_10m`


In [23]:
features = [
    'hour',
    'is_weekend',
    'trip_distance_km',
    'temperature_2m',
    'relative_humidity_2m',
    'precipitation',
    'wind_speed_10m'
]

target = 'trip_duration_min'

ml_df = df[['started_at'] + features + [target]].dropna().copy()

print("ML dataset shape:", ml_df.shape)
print("\nMissing values:")
print(ml_df.isnull().sum())


ML dataset shape: (0, 9)

Missing values:
started_at              0
hour                    0
is_weekend              0
trip_distance_km        0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
trip_duration_min       0
dtype: int64


In [24]:
# Time-based 80/20 split

split_index = int(len(ml_df) * 0.80)

train_df = ml_df.iloc[:split_index].copy()
test_df = ml_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining period:")
print(train_df['started_at'].min(), "to", train_df['started_at'].max())

print("\nTesting period:")
print(test_df['started_at'].min(), "to", test_df['started_at'].max())


Train shape: (0, 9)
Test shape: (0, 9)

Training period:
NaT to NaT

Testing period:
NaT to NaT


In [25]:
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (0, 7)
X_test: (0, 7)
y_train: (0,)
y_test: (0,)


## 2. Baseline Model

The baseline predicts the mean trip duration from the training set for every test observation.


In [28]:
# Mean Baseline

baseline_prediction = np.repeat(
    y_train.mean(),
    len(y_test)
)

if len(y_test) == 0:
    print("Error: y_test is empty.")
    print("Check the train/test split before calculating metrics.")
else:
    baseline_mae = mean_absolute_error(y_test, baseline_prediction)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_prediction))
    baseline_r2 = r2_score(y_test, baseline_prediction)

    print("Mean Baseline")
    print("----------------")
    print("MAE :", round(baseline_mae, 2))
    print("RMSE:", round(baseline_rmse, 2))
    print("R²  :", round(baseline_r2, 4))

Error: y_test is empty.
Check the train/test split before calculating metrics.


## 3. Linear Regression

In [29]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model trained successfully.")


ValueError: Found array with 0 sample(s) (shape=(0, 7)) while a minimum of 1 is required by LinearRegression.

In [ ]:
# Predictions

y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

print(comparison.head(10))


## 4. Model Evaluation

Metrics:
- MAE
- RMSE
- R²
- MAPE

Note: MAPE can become unstable when actual trip duration is zero or very close to zero, so we calculate it only for positive actual values.


In [ ]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

positive_mask = y_test > 0

mape = np.mean(
    np.abs(
        (y_test[positive_mask] - y_pred[positive_mask])
        / y_test[positive_mask]
    )
) * 100

print("Linear Regression Results")
print("-------------------------")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))
print("MAPE:", round(mape, 2), "%")


In [ ]:
# Compare baseline and Linear Regression

results = pd.DataFrame({
    'Model': ['Mean Baseline', 'Linear Regression'],
    'MAE': [baseline_mae, mae],
    'RMSE': [baseline_rmse, rmse],
    'R2': [baseline_r2, r2]
})

print(results)


## 5. Coefficient Analysis

Coefficients show the direction and magnitude of the fitted linear relationship, holding the other included features constant.


In [ ]:
coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_
})

coefficients = coefficients.sort_values(
    'Coefficient',
    ascending=False
)

print(coefficients)


In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=coefficients,
    x='Coefficient',
    y='Feature'
)

plt.title('Linear Regression Coefficients')
plt.xlabel('Coefficient')
plt.ylabel('Feature')
plt.show()


## 6. Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.2
)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle='--'
)

plt.xlabel('Actual Trip Duration (minutes)')
plt.ylabel('Predicted Trip Duration (minutes)')
plt.title('Actual vs Predicted Trip Duration')

plt.show()


## 7. Residual Analysis

In [ ]:
residuals = y_test - y_pred

print("Residual mean:", round(residuals.mean(), 4))
print("Residual median:", round(residuals.median(), 4))


In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    residuals,
    bins=50,
    kde=True
)

plt.title('Residual Distribution')
plt.xlabel('Residual (Actual - Predicted)')
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(
    y_pred,
    residuals,
    alpha=0.2
)

plt.axhline(
    0,
    linestyle='--'
)

plt.xlabel('Predicted Trip Duration')
plt.ylabel('Residual')
plt.title('Residuals vs Predicted Values')

plt.show()


## 8. Final ML Summary

Record the actual results after running the notebook.

- Baseline MAE: ___
- Baseline RMSE: ___
- Baseline R²: ___
- Linear Regression MAE: ___
- Linear Regression RMSE: ___
- Linear Regression R²: ___
- Linear Regression MAPE: ___

### Conclusion

Compare Linear Regression with the baseline and describe where the model performs well or poorly. Do not claim that a model is good or bad without referring to the measured results.
